In [37]:
%useLatestDescriptors
%use dataframe
@file:DependsOn("com.github.doyaaaaaken:kotlin-csv-jvm:1.7.0")

import com.github.doyaaaaaken.kotlincsv.dsl.csvWriter
import io.github.oshai.kotlinlogging.KotlinLogging.logger
import kotlin.reflect.full.declaredMemberProperties
import java.nio.file.Paths
import java.util.Locale
import kotlin.io.path.Path

enum class Mode { FLAT, RANDOM }
enum class Algorithm(val shortName: String) {
    FROMBACK("tsprcs"),
    DISTANCE("tsprce"),
    SPARSITY("tsprcs"),
    OP("op"),
}

val percentageFraction = 1
val colsWithoutPercentages = "OP"
val gradient = 0.1

val mode = Mode.FLAT
val algorithm = Algorithm.OP

val fileName = "comparison_${mode.name.lowercase()}.csv"

val relativePath = "/op-solver-strict/results/elimination/"
val navigationPath = Paths.get(System.getProperty("user.dir"), "../../../..").normalize().toAbsolutePath().toString()

val path = Paths.get(navigationPath, relativePath, fileName).toString()

var df = DataFrame.readCsv(path)
df

instance,TSPrfb,TSPrce,OP
eil101,31.000000,31.000000,34.000000
gil262,67.000000,68.000000,72.000000
pr299,74.000000,73.000000,77.000000
lin318,98.000000,94.000000,94.000000
rd400,102.000000,102.000000,104.000000
d493,168.000000,142.000000,144.000000
u574,154.000000,156.000000,158.000000
u724,195.000000,189.000000,193.000000
pcb1173,291.000000,290.000000,292.000000
fl1400,435.000000,423.000000,522.000000


In [38]:
val bestValues = df.convert { all() }.perRowCol { row, col ->
    if (col[row] is String) {
        0
    } else {
        (col[row] as Double).toInt()
    }
}.map { row ->
    row.rowMaxOf<Int>()
}

bestValues

[34, 72, 77, 98, 104, 168, 158, 195, 292, 522, 618]

In [39]:

 //max(maxRevenueDif,0.0)

val rowMaxValues = bestValues.mapIndexed { index, resultMax -> max(resultMax, (df["OP"][index] as Double).toInt()) }

rowMaxValues

[34, 72, 77, 98, 104, 168, 158, 195, 292, 522, 618]

In [40]:
val rowMinValues = df.map { row ->
    row.rowMinOfOrNull<Double>()
}.map { row -> row!!.toInt()}
rowMinValues

[31, 67, 73, 94, 102, 142, 154, 189, 290, 423, 584]

In [41]:

val revenueDif = rowMaxValues.mapIndexed { index, maxEntry ->
    val minEntry = df[index].rowMinOf<Double>()
    (maxEntry - minEntry!!).toDouble() / maxEntry.toDouble()
}

fun getSaturation(gradient: Double, maxValue: Int, value: Int): String {
    return min(max((100 - ((maxValue - value) / (maxValue * gradient) * 100)), 0.0), 100.0).toInt().toString()
}

fun calculatePercentage(refValue: Int, compValue: Int): Double {
    return ((compValue.toDouble() - refValue.toDouble()) / refValue.toDouble())
}

fun formatePercentage(value: Double): String {
    return "${String.format(Locale.US, "%+.${percentageFraction}f", value * 100)}\\%"
}

fun getPercentage(row: DataRow<*>, compValue: Int): String {
    val refValue = (df.get(colsWithoutPercentages)[row] as Double).toInt()
    val percentage = calculatePercentage(refValue, compValue)
    return "{\\tiny${formatePercentage(percentage)}}"
}

val footer = df.convert { all() }.perRowCol { row, col ->
    if (col.name() == colsWithoutPercentages || col[row] is String) {
        10000.0
    } else {
        val refValue = (df.get(colsWithoutPercentages)[row] as Double).toInt()
        calculatePercentage(refValue, (col[row] as Double).toInt())
    }
}.mean().values().mapIndexed { index, it ->
    if (index == 0) {
        "avg diff"
    } else if (it is Double && it > 100.0) {
        "-"
    } else if (it is Double) {
        formatePercentage(it)
    } else {
        "${it.toString()}\\%"
    }
}.toList()
footer

[avg diff, -2.1\%, -4.7\%, -]

In [42]:

val stringdf = df.convert { all() }.perRowCol { row, col ->
    if (col[row] is String) {
        col[row].toString().split("-").first()
    } else if (col.name() == colsWithoutPercentages) {
        val value = (col[row] as Double).toInt()
        val maxValue = rowMaxValues[row.index()]
        val saturation = getSaturation(gradient, maxValue, value)
        if (bestValues[row.index()] == value) {
            "\\cellcolor{cyan!$saturation} \\textbf{$value*}"
        } else {
            "\\cellcolor{cyan!$saturation} $value"
        }
    } else {
        val value = (col[row] as Double).toInt()
        val maxValue = rowMaxValues[row.index()]
        val saturation = getSaturation(gradient, maxValue, value)
        val percentage = getPercentage(row, value)
        if (bestValues[row.index()] == value) {
            "\\cellcolor{cyan!$saturation} \\textbf{$value*}$percentage"
        } else {
            "\\cellcolor{cyan!$saturation} $value$percentage"
        }
    }
}
stringdf

instance,TSPrfb,TSPrce,OP
eil101,\cellcolor{cyan!11} 31{\tiny-8.8\%},\cellcolor{cyan!11} 31{\tiny-8.8\%},\cellcolor{cyan!100} \textbf{34*}
gil262,\cellcolor{cyan!30} 67{\tiny-6.9\%},\cellcolor{cyan!44} 68{\tiny-5.6\%},\cellcolor{cyan!100} \textbf{72*}
pr299,\cellcolor{cyan!61} 74{\tiny-3.9\%},\cellcolor{cyan!48} 73{\tiny-5.2\%},\cellcolor{cyan!100} \textbf{77*}
lin318,\cellcolor{cyan!100} \textbf{98*}{\ti...,\cellcolor{cyan!59} 94{\tiny+0.0\%},\cellcolor{cyan!59} 94
rd400,\cellcolor{cyan!80} 102{\tiny-1.9\%},\cellcolor{cyan!80} 102{\tiny-1.9\%},\cellcolor{cyan!100} \textbf{104*}
d493,\cellcolor{cyan!100} \textbf{168*}{\t...,\cellcolor{cyan!0} 142{\tiny-1.4\%},\cellcolor{cyan!0} 144
u574,\cellcolor{cyan!74} 154{\tiny-2.5\%},\cellcolor{cyan!87} 156{\tiny-1.3\%},\cellcolor{cyan!100} \textbf{158*}
u724,\cellcolor{cyan!100} \textbf{195*}{\t...,\cellcolor{cyan!69} 189{\tiny-2.1\%},\cellcolor{cyan!89} 193
pcb1173,\cellcolor{cyan!96} 291{\tiny-0.3\%},\cellcolor{cyan!93} 290{\tiny-0.7\%},\cellcolor{cyan!100} \textbf{292*}
fl1400,\cellcolor{cyan!0} 435{\tiny-16.7\%},\cellcolor{cyan!0} 423{\tiny-19.0\%},\cellcolor{cyan!100} \textbf{522*}


In [43]:
fun getLatexTable(formating: String, amountColumns: String, title: String, header: String, label: String, caption: String, body: String): String {
    return """
    \begin{table}[]
        \vspace{2em}
        \begin{adjustbox}{center}
            \begin{tabular}{ $formating  }
                \hline
                \multicolumn{$amountColumns}{|c|}{$title} \\
                \hline
                    $header \\
                \hline
                    $body
                \hline
            \end{tabular}
        \end{adjustbox}
        \caption{$caption}
        \label{$label}
    \end{table}
         """
}

val shortAlgString = when (algorithm) {
    Algorithm.FROMBACK -> "TSPrfb"
    Algorithm.DISTANCE -> "TSPrce"
    Algorithm.SPARSITY -> "TSPrcs"
    Algorithm.OP -> "OP"
}
val mediumAlgString = when (algorithm) {
    Algorithm.FROMBACK -> "cluster removal from back"
    Algorithm.DISTANCE -> "cluster removal based on distance"
    Algorithm.SPARSITY -> "cluster removal based on sparsity"
    Algorithm.OP -> "implicit cluster removal"
}

val amountColumns = df.columns().size.toString()
val formating = "|"+ df.columns().joinToString(separator = "") { "p{1.9cm}|" }
val header = df.columnNames().joinToString(separator = " & ")
val label = "tab:elim:${mode.name.lowercase()}"
val title = "emimination methods ${mode.name.lowercase()}."
//Parameter run for $R'$ using cluster removal from back with a budget of $\gamma = 0.5$. The percentage value refers to the mean revenue increase compared to $e^{blr}$. The highest revenue of an instance has 100\% saturation decreasing to 0\% at 70\% of the maximum. $\textbf{*}$ refers to the best mean revenue.
val caption = "Emimination method comparison for \$R' = 0.5\$ and \$\\alpha = 0.25\$ with $\\gamma = 0.5\$. The percentage value refers to the mean revenue increase compared to \$e^{bl${mode.name.lowercase().first().toString()}}$. The highest revenue of an instance has 100\\% saturation decreasing to 0\\% at ${(100 + gradient * -100).toInt()}\\% of the maximum revenue. The larges revenue value is referenced by \$\\textbf{*}\$."

val body = stringdf.rows().joinToString(separator = " \\\\ \n") { row ->
    row.values().joinToString(separator = " & ") {
        it.toString()
    }
} + " \\\\ \\hline " + footer.joinToString(separator = " & ") {
    it.toString()
} + " \\\\"

getLatexTable(formating, amountColumns, title, header, label, caption, body)


    \begin{table}[]
        \vspace{2em}
        \begin{adjustbox}{center}
            \begin{tabular}{ |p{1.9cm}|p{1.9cm}|p{1.9cm}|p{1.9cm}|  }
                \hline
                \multicolumn{4}{|c|}{emimination methods flat.} \\
                \hline
                    instance & TSPrfb & TSPrce & OP \\
                \hline
                    eil101 & \cellcolor{cyan!11} 31{\tiny-8.8\%} & \cellcolor{cyan!11} 31{\tiny-8.8\%} & \cellcolor{cyan!100} \textbf{34*} \\ 
gil262 & \cellcolor{cyan!30} 67{\tiny-6.9\%} & \cellcolor{cyan!44} 68{\tiny-5.6\%} & \cellcolor{cyan!100} \textbf{72*} \\ 
pr299 & \cellcolor{cyan!61} 74{\tiny-3.9\%} & \cellcolor{cyan!48} 73{\tiny-5.2\%} & \cellcolor{cyan!100} \textbf{77*} \\ 
lin318 & \cellcolor{cyan!100} \textbf{98*}{\tiny+4.3\%} & \cellcolor{cyan!59} 94{\tiny+0.0\%} & \cellcolor{cyan!59} 94 \\ 
rd400 & \cellcolor{cyan!80} 102{\tiny-1.9\%} & \cellcolor{cyan!80} 102{\tiny-1.9\%} & \cellcolor{cyan!100} \textbf{104*} \\ 
d493 & \cellcolor{cyan!100}